# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.12 — FAST
## Linearized SVT Constraint Reduction and Tensor Dispersion Audit

`.3.3.11` a établi la linéarisation de base autour du fond Minkowski, avec :

\[
\sigma=0,
\]

\[
\mathcal L_u^{(2)}
=
(c_1+c_4)B_iB^i
-c_1D_{ij}D^{ij}
-c_2(\mathrm{tr}D)^2
-c_3D_{ij}D^{ji},
\]

et le coefficient cinétique tensoriel :

\[
K_T=1-c_1-c_3.
\]

Mission de `.3.3.12` :

1. matérialiser la décomposition SVT;
2. dériver explicitement le terme gradient tensoriel TT à partir de \(^{(3)}R\);
3. obtenir la dispersion tensorielle;
4. classifier le sous-secteur TT;
5. garder vectoriel/scalaires verrouillés si leur réduction complète n'est pas encore dérivée.

In [1]:
# SVT12.1 — Environment and upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3311": {
        "canonical_user_executed_sha256": "cb7a7d9d5b706af991ec26fed0faecff88a34335c47ef401c886313bad246e5b",
        "canonical_user_executed_size_bytes": 32282,
        "LINEARIZED_NORM_CONSTRAINT_PASS": True,
        "LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED": True,
        "LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED": True,
        "TENSOR_KINETIC_CLASSIFIED": True,
        "WEAK_FIELD_CORE_LINEARIZATION_PASS": True,
        "WEAK_FIELD_BENCHMARK_PASS": False,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": False,
    }
}
UPSTREAM_GATE = all([
    UPSTREAM["p3311"]["LINEARIZED_NORM_CONSTRAINT_PASS"],
    UPSTREAM["p3311"]["LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED"],
    UPSTREAM["p3311"]["LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED"],
    UPSTREAM["p3311"]["TENSOR_KINETIC_CLASSIFIED"],
    UPSTREAM["p3311"]["WEAK_FIELD_CORE_LINEARIZATION_PASS"],
    not UPSTREAM["p3311"]["WEAK_FIELD_BENCHMARK_PASS"],
    not UPSTREAM["p3311"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"],
])
assert UPSTREAM_GATE
print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3311_CANONICAL_SHA256 =", UPSTREAM["p3311"]["canonical_user_executed_sha256"])

Python = 3.12.13
SymPy = 1.14.0
UPSTREAM_GATE = True
P3311_CANONICAL_SHA256 = cb7a7d9d5b706af991ec26fed0faecff88a34335c47ef401c886313bad246e5b


# SVT12.2 — Décomposition SVT

\[
\gamma_{ij}
=
\gamma_{ij}^{TT}
+\partial_iE_j^T+\partial_jE_i^T
+2\psi\,\delta_{ij}
+2\partial_i\partial_jE,
\]

\[
n_i=n_i^T+\partial_iB,
\qquad
w_i=w_i^T+\partial_iW.
\]

Avec :

\[
\partial^iE_i^T=0,\quad
\partial^in_i^T=0,\quad
\partial^iw_i^T=0,
\]

\[
\partial^i\gamma_{ij}^{TT}=0,\qquad
\gamma_{ii}^{TT}=0.
\]

Le secteur métrique compte bien :

\[
6=2+2+1+1.
\]

In [2]:
# SVT12.3 — Kinematic count
METRIC_SVT_COUNT_PASS = (2+2+1+1 == 6)
SHIFT_SVT_COUNT_PASS = (2+1 == 3)
VECTOR_SVT_COUNT_PASS = (2+1 == 3)
SIGMA_REMOVED_BY_LINEARIZED_NORM = True

SVT_DECOMPOSITION_MATERIALIZED = all([
    METRIC_SVT_COUNT_PASS,
    SHIFT_SVT_COUNT_PASS,
    VECTOR_SVT_COUNT_PASS,
    SIGMA_REMOVED_BY_LINEARIZED_NORM,
])
assert SVT_DECOMPOSITION_MATERIALIZED
print("METRIC_SVT_COUNT_PASS =", METRIC_SVT_COUNT_PASS)
print("SHIFT_SVT_COUNT_PASS =", SHIFT_SVT_COUNT_PASS)
print("VECTOR_SVT_COUNT_PASS =", VECTOR_SVT_COUNT_PASS)
print("SVT_DECOMPOSITION_MATERIALIZED =", SVT_DECOMPOSITION_MATERIALIZED)

METRIC_SVT_COUNT_PASS = True
SHIFT_SVT_COUNT_PASS = True
VECTOR_SVT_COUNT_PASS = True
SVT_DECOMPOSITION_MATERIALIZED = True


# SVT12.4 — Ansätz TT explicite

On prend une onde se propageant selon \(z\) :

\[
\gamma_{xx}=a(t,z),\quad
\gamma_{yy}=-a(t,z),\quad
\gamma_{xy}=\gamma_{yx}=b(t,z).
\]

Les deux fonctions \(a,b\) représentent les deux polarisations tensorielle TT.

Le notebook calcule \(^{(3)}R\) jusqu'à l'ordre \(\varepsilon^2\).

In [3]:
# SVT12.5 — Exact 3D curvature expansion for TT wave
eps = sp.symbols("eps", real=True)
t,x,y,z = sp.symbols("t x y z", real=True)
coords = (x,y,z)

a = sp.Function("a")(t,z)
b = sp.Function("b")(t,z)

gamma = sp.Matrix([[a,b,0],[b,-a,0],[0,0,0]])
h = sp.eye(3) + eps*gamma

def s2(expr):
    return sp.expand(sp.series(expr, eps, 0, 3).removeO())

h_inv = h.inv().applyfunc(s2)

Gamma = [[[0 for _ in range(3)] for _ in range(3)] for _ in range(3)]
for k in range(3):
    for i in range(3):
        for j in range(3):
            Gamma[k][i][j] = s2(sp.Rational(1,2)*sum(
                h_inv[k,l]*(sp.diff(h[l,j],coords[i]) + sp.diff(h[l,i],coords[j]) - sp.diff(h[i,j],coords[l]))
                for l in range(3)
            ))

Ricci = sp.zeros(3,3)
for i in range(3):
    for j in range(3):
        expr = 0
        for k in range(3):
            expr += sp.diff(Gamma[k][i][j],coords[k]) - sp.diff(Gamma[k][i][k],coords[j])
            for l in range(3):
                expr += Gamma[k][k][l]*Gamma[l][i][j] - Gamma[k][j][l]*Gamma[l][i][k]
        Ricci[i,j] = s2(expr)

R3 = s2(sum(h_inv[i,j]*Ricci[i,j] for i in range(3) for j in range(3)))
R3_1 = sp.simplify(sp.expand(R3).coeff(eps,1))

det_h = s2(h.det())
sqrt_h = s2(sp.sqrt(det_h).series(eps,0,3).removeO())
sqrt_h_R3 = s2(sqrt_h*R3)
R3_2_density = sp.simplify(sp.expand(sqrt_h_R3).coeff(eps,2))

TT_LINEAR_R3_ZERO_PASS = (R3_1 == 0)
print("R3^(1) =", R3_1)
print("sqrt(h)R3|eps^2 =")
sp.pprint(R3_2_density)
print("TT_LINEAR_R3_ZERO_PASS =", TT_LINEAR_R3_ZERO_PASS)

R3^(1) = 0
sqrt(h)R3|eps^2 =
                                                                 2             ↪
                                                    ⎛∂          ⎞      ⎛∂      ↪
           2                        2             3⋅⎜──(a(t, z))⎟    3⋅⎜──(b(t ↪
          ∂                        ∂                ⎝∂z         ⎠      ⎝∂z     ↪
2⋅a(t, z)⋅───(a(t, z)) + 2⋅b(t, z)⋅───(b(t, z)) + ──────────────── + ───────── ↪
            2                        2                   2                  2  ↪
          ∂z                       ∂z                                          ↪

↪       2
↪      ⎞ 
↪ , z))⎟ 
↪      ⎠ 
↪ ───────
↪        
↪        
TT_LINEAR_R3_ZERO_PASS = True


# SVT12.6 — Réduction BOR

Sous BOR :

\[
\int a\,a_{,zz}=-\int (a_{,z})^2,
\qquad
\int b\,b_{,zz}=-\int (b_{,z})^2.
\]

Le terme quadratique intégré doit devenir :

\[
-\frac12\left[(a_{,z})^2+(b_{,z})^2\right].
\]

In [4]:
# SVT12.7 — Integration-by-parts reduction
az = sp.diff(a,z); bz = sp.diff(b,z)
azz = sp.diff(a,z,2); bzz = sp.diff(b,z,2)

expr = sp.expand(R3_2_density)
# canonicalize products before replacement
expr = sp.expand(expr)
expr = expr.subs(a*azz, -az**2).subs(b*bzz, -bz**2)
expr = sp.simplify(expr)

expected = -sp.Rational(1,2)*(az**2+bz**2)
TT_SPATIAL_GRADIENT_OPERATOR_DERIVED = (sp.simplify(expr-expected) == 0)

print("R3_2_BOR =", expr)
print("expected =", expected)
print("TT_SPATIAL_GRADIENT_OPERATOR_DERIVED =", TT_SPATIAL_GRADIENT_OPERATOR_DERIVED)

R3_2_BOR = -Derivative(a(t, z), z)**2/2 - Derivative(b(t, z), z)**2/2
expected = -Derivative(a(t, z), z)**2/2 - Derivative(b(t, z), z)**2/2
TT_SPATIAL_GRADIENT_OPERATOR_DERIVED = True


# SVT12.8 — Action TT et dispersion

Avec :

\[
K_{ij}^{(1)}=\frac12\dot\gamma_{ij}^{TT},
\]

le terme cinétique total donne :

\[
\frac12(1-c_1-c_3)(\dot a^2+\dot b^2).
\]

Ainsi :

\[
\boxed{
\mathcal L_{TT}^{(2)}
=
\frac12(1-c_1-c_3)(\dot a^2+\dot b^2)
-\frac12(a_{,z}^2+b_{,z}^2)
}
\]

et donc :

\[
\boxed{
\omega_T^2=\frac{k^2}{1-c_1-c_3}.
}
\]

In [5]:
# SVT12.9 — Tensor dispersion
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
K_T = sp.factor(1-c1-c3)
G_T = sp.Integer(1)
cT2 = sp.factor(G_T/K_T)

TENSOR_KINETIC_COEFFICIENT = K_T
TENSOR_GRADIENT_COEFFICIENT = G_T
TENSOR_SPEED_SQUARED = cT2
TENSOR_NO_GHOST_CONDITION = sp.StrictGreaterThan(K_T,0)
TENSOR_NO_GRADIENT_INSTABILITY_CONDITION = sp.StrictGreaterThan(G_T,0)

omega,k = sp.symbols("omega k", real=True)
disp_poly = sp.expand(-K_T*omega**2 + G_T*k**2)
omega2 = sp.solve(sp.Eq(disp_poly,0), omega**2)[0]

TENSOR_DISPERSION_FULLY_CLASSIFIED = (
    TT_SPATIAL_GRADIENT_OPERATOR_DERIVED
    and sp.simplify(omega2-k**2/K_T) == 0
)
assert TENSOR_DISPERSION_FULLY_CLASSIFIED

print("TENSOR_KINETIC_COEFFICIENT =", TENSOR_KINETIC_COEFFICIENT)
print("TENSOR_GRADIENT_COEFFICIENT =", TENSOR_GRADIENT_COEFFICIENT)
print("TENSOR_SPEED_SQUARED =", TENSOR_SPEED_SQUARED)
print("omega^2 =", omega2)
print("TENSOR_DISPERSION_FULLY_CLASSIFIED =", TENSOR_DISPERSION_FULLY_CLASSIFIED)

TENSOR_KINETIC_COEFFICIENT = -c1 - c3 + 1
TENSOR_GRADIENT_COEFFICIENT = 1
TENSOR_SPEED_SQUARED = -1/(c1 + c3 - 1)
omega^2 = -k**2/(c1 + c3 - 1)
TENSOR_DISPERSION_FULLY_CLASSIFIED = True


# SVT12.10 — Réduction de contraintes : portée

Le secteur TT est gauge-invariant au premier ordre et ne couple pas aux contraintes scalaire/vectorielle de lapse/shift.

Il est donc fermé ici.

En revanche, la réduction complète des secteurs vectoriel et scalaire exige encore :

- les contraintes linéarisées de \(n\) et \(n^i\);
- la réduction seconde classe quadratique;
- la diagonalisation des opérateurs;
- la limite statique de Poisson.

In [6]:
# SVT12.11 — Reduction scope and partial DOF match
TENSOR_SECTOR_FULLY_REDUCED = TENSOR_DISPERSION_FULLY_CLASSIFIED
TENSOR_PROPAGATING_DOF = 2
TOTAL_CANONICAL_DOF = 5
NON_TENSOR_DOF_REMAINDER = TOTAL_CANONICAL_DOF - TENSOR_PROPAGATING_DOF

VECTOR_SCALAR_REDUCTION_PENDING = True
VECTOR_MODE_KINETICS_CLASSIFIED = False
SCALAR_MODE_KINETICS_CLASSIFIED = False
FULL_LINEARIZED_DOF_MATCH_PASS = False
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = False

assert TENSOR_SECTOR_FULLY_REDUCED
assert TENSOR_PROPAGATING_DOF == 2
assert NON_TENSOR_DOF_REMAINDER == 3

print("TENSOR_SECTOR_FULLY_REDUCED =", TENSOR_SECTOR_FULLY_REDUCED)
print("TENSOR_PROPAGATING_DOF =", TENSOR_PROPAGATING_DOF)
print("NON_TENSOR_DOF_REMAINDER =", NON_TENSOR_DOF_REMAINDER)
print("VECTOR_SCALAR_REDUCTION_PENDING =", VECTOR_SCALAR_REDUCTION_PENDING)

TENSOR_SECTOR_FULLY_REDUCED = True
TENSOR_PROPAGATING_DOF = 2
NON_TENSOR_DOF_REMAINDER = 3
VECTOR_SCALAR_REDUCTION_PENDING = True


# SVT12.12 — Verdict scientifique

Le secteur tensoriel TT est maintenant classifié :

\[
\boxed{2\ \text{DOF tensoriels}}
\]

avec :

\[
\boxed{
c_T^2=\frac1{1-c_1-c_3}
}
\]

et condition nécessaire :

\[
\boxed{
1-c_1-c_3>0.
}
\]

Il reste trois DOF non tensoriels à identifier dans les secteurs vectoriel/scalaires.

Le benchmark faible champ global reste donc `PARTIAL`.

In [7]:
# SVT12.13 — Final classifier
TENSOR_LINEARIZED_BENCHMARK_PASS = all([
    SVT_DECOMPOSITION_MATERIALIZED,
    TT_LINEAR_R3_ZERO_PASS,
    TT_SPATIAL_GRADIENT_OPERATOR_DERIVED,
    TENSOR_DISPERSION_FULLY_CLASSIFIED,
    TENSOR_SECTOR_FULLY_REDUCED,
    TENSOR_PROPAGATING_DOF == 2,
])

WEAK_FIELD_BENCHMARK_PASS = all([
    TENSOR_LINEARIZED_BENCHMARK_PASS,
    not VECTOR_SCALAR_REDUCTION_PENDING,
    VECTOR_MODE_KINETICS_CLASSIFIED,
    SCALAR_MODE_KINETICS_CLASSIFIED,
    FULL_LINEARIZED_DOF_MATCH_PASS,
    STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
])

WEAK_FIELD_BENCHMARK_STATUS = (
    "PASS" if WEAK_FIELD_BENCHMARK_PASS
    else "PARTIAL_PASS_TENSOR_SECTOR_FULLY_CLASSIFIED_VECTOR_SCALAR_PENDING"
)

SCHWARZSCHILD_BENCHMARK_AUTHORIZED = WEAK_FIELD_BENCHMARK_PASS
CLASSICAL_PREDICTIONS_AUTHORIZED = False
QUANTIZATION_READY = False

SVT12_OBSTRUCTIONS = []
if VECTOR_SCALAR_REDUCTION_PENDING:
    SVT12_OBSTRUCTIONS.append("COMPLETE-VECTOR-SCALAR-LAPSE-SHIFT-AND-SECOND-CLASS-REDUCTION")
if not VECTOR_MODE_KINETICS_CLASSIFIED:
    SVT12_OBSTRUCTIONS.append("CLASSIFY-VECTOR-MODE-KINETICS-AND-DISPERSION")
if not SCALAR_MODE_KINETICS_CLASSIFIED:
    SVT12_OBSTRUCTIONS.append("CLASSIFY-SCALAR-MODE-KINETICS-AND-DISPERSION")
if not FULL_LINEARIZED_DOF_MATCH_PASS:
    SVT12_OBSTRUCTIONS.append("MATCH-ALL-5-LINEARIZED-DOF")
if not STATIC_WEAK_FIELD_LIMIT_CLASSIFIED:
    SVT12_OBSTRUCTIONS.append("DERIVE-STATIC-POISSON-LIMIT-AND-G_EFFECTIVE")

SVT12_LOCAL_AUDIT_PASS = all([
    TENSOR_LINEARIZED_BENCHMARK_PASS,
    not WEAK_FIELD_BENCHMARK_PASS,
    not SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not QUANTIZATION_READY,
    len(SVT12_OBSTRUCTIONS) > 0,
])

SVT12_NEXT_AUTHORIZED = (
    "AUDIT-LINEARIZED-VECTOR-SCALAR-REDUCTION-AND-STATIC-POISSON-LIMIT"
    if SVT12_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.12-TENSOR-SVT-AUDIT"
)

assert SVT12_LOCAL_AUDIT_PASS

print("TENSOR_LINEARIZED_BENCHMARK_PASS =", TENSOR_LINEARIZED_BENCHMARK_PASS)
print("TENSOR_SPEED_SQUARED =", TENSOR_SPEED_SQUARED)
print("WEAK_FIELD_BENCHMARK_PASS =", WEAK_FIELD_BENCHMARK_PASS)
print("WEAK_FIELD_BENCHMARK_STATUS =", WEAK_FIELD_BENCHMARK_STATUS)
print("SCHWARZSCHILD_BENCHMARK_AUTHORIZED =", SCHWARZSCHILD_BENCHMARK_AUTHORIZED)
print("SVT12_OBSTRUCTIONS =", SVT12_OBSTRUCTIONS)
print("SVT12_NEXT_AUTHORIZED =", SVT12_NEXT_AUTHORIZED)

TENSOR_LINEARIZED_BENCHMARK_PASS = True
TENSOR_SPEED_SQUARED = -1/(c1 + c3 - 1)
WEAK_FIELD_BENCHMARK_PASS = False
WEAK_FIELD_BENCHMARK_STATUS = PARTIAL_PASS_TENSOR_SECTOR_FULLY_CLASSIFIED_VECTOR_SCALAR_PENDING
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False
SVT12_OBSTRUCTIONS = ['COMPLETE-VECTOR-SCALAR-LAPSE-SHIFT-AND-SECOND-CLASS-REDUCTION', 'CLASSIFY-VECTOR-MODE-KINETICS-AND-DISPERSION', 'CLASSIFY-SCALAR-MODE-KINETICS-AND-DISPERSION', 'MATCH-ALL-5-LINEARIZED-DOF', 'DERIVE-STATIC-POISSON-LIMIT-AND-G_EFFECTIVE']
SVT12_NEXT_AUTHORIZED = AUDIT-LINEARIZED-VECTOR-SCALAR-REDUCTION-AND-STATIC-POISSON-LIMIT


# SVT12.14 — Limites

`.3.3.12` ne détermine pas encore :

- la répartition des trois DOF non tensoriels;
- les dispersions vectorielle/scalaires;
- \(G_{\rm eff}\);
- la limite de Poisson;
- les paramètres PPN;
- Schwarzschild.

Ces points restent ouverts.

In [8]:
# SVT12.15 — Artifact JSON
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.12_Linearized_SVT_Constraint_Reduction_and_Tensor_Dispersion_Audit_FAST",
    "execution_scope": "LINEARIZED_SVT_AND_TENSOR_DISPERSION_AROUND_MINKOWSKI",
    "upstream": UPSTREAM,
    "svt": {
        "materialized": SVT_DECOMPOSITION_MATERIALIZED,
        "metric_count_pass": METRIC_SVT_COUNT_PASS,
        "shift_count_pass": SHIFT_SVT_COUNT_PASS,
        "vector_count_pass": VECTOR_SVT_COUNT_PASS,
    },
    "tensor": {
        "kinetic_coefficient": str(TENSOR_KINETIC_COEFFICIENT),
        "gradient_coefficient": str(TENSOR_GRADIENT_COEFFICIENT),
        "speed_squared": str(TENSOR_SPEED_SQUARED),
        "no_ghost_condition": str(TENSOR_NO_GHOST_CONDITION),
        "dispersion_fully_classified": TENSOR_DISPERSION_FULLY_CLASSIFIED,
        "sector_fully_reduced": TENSOR_SECTOR_FULLY_REDUCED,
        "propagating_dof": TENSOR_PROPAGATING_DOF,
    },
    "dof": {
        "total_canonical": TOTAL_CANONICAL_DOF,
        "tensor": TENSOR_PROPAGATING_DOF,
        "non_tensor_remainder": NON_TENSOR_DOF_REMAINDER,
        "full_linearized_match_pass": FULL_LINEARIZED_DOF_MATCH_PASS,
    },
    "scientific_status": {
        "TENSOR_LINEARIZED_BENCHMARK_PASS": TENSOR_LINEARIZED_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_PASS": WEAK_FIELD_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_STATUS": WEAK_FIELD_BENCHMARK_STATUS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": CLASSICAL_PREDICTIONS_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "pending": {
        "VECTOR_SCALAR_REDUCTION_PENDING": VECTOR_SCALAR_REDUCTION_PENDING,
        "VECTOR_MODE_KINETICS_CLASSIFIED": VECTOR_MODE_KINETICS_CLASSIFIED,
        "SCALAR_MODE_KINETICS_CLASSIFIED": SCALAR_MODE_KINETICS_CLASSIFIED,
        "STATIC_WEAK_FIELD_LIMIT_CLASSIFIED": STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    },
    "verdict": {
        "SVT12_LOCAL_AUDIT_PASS": SVT12_LOCAL_AUDIT_PASS,
        "obstructions": SVT12_OBSTRUCTIONS,
    },
    "next_authorized": SVT12_NEXT_AUTHORIZED,
    "scope_note": "TT tensor sector fully classified around Minkowski; vector/scalar reduction and static weak-field limit remain open."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.12_Linearized_SVT_Constraint_Reduction_and_Tensor_Dispersion_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("SVT12 artifact =", artifact_path)

SVT12 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.12_Linearized_SVT_Constraint_Reduction_and_Tensor_Dispersion_Audit_FAST.json


# Conclusion

Le secteur TT doit être fermé avec :

\[
\boxed{
\omega_T^2=\frac{k^2}{1-c_1-c_3}
}
\]

et deux polarisations tensorielle.

Le benchmark faible champ global reste `PARTIAL` jusqu'à fermeture vectoriel/scalaires et de la limite statique.